# 배포 아키텍처

```mermaid
flowchart TB
    subgraph Client["프론트엔드 (React)"]
        FE["RegiHelper UI"]
    end

    subgraph API["app/main.py — FastAPI"]
        Health["GET /health"]
        Ask["POST /ask (동기)"]
        CreateJob["POST /jobs"]
        ReadJob["GET /jobs/&#123;job_id&#125;"]
    end

    subgraph Async["비동기 잡 처리"]
        Jobs["app/jobs.py"]
        Cache[("cache.py\nDynamoDB\nrealestate-cache\n완전일치 캐시, TTL 24h")]
        SQS[("Amazon SQS\nJOB_QUEUE_URL")]
        DDB[("DynamoDB\nrealestate-jobs\nTTL 1h")]
        Worker["app/worker.py\n백그라운드 스레드\n롱폴링(20s)"]
    end

    subgraph Core["app/supervisor.py — LangGraph"]
        Planner["planner\n(gpt-4o-mini, structured output)\nPlan&#123;next, use_decomp, reason&#125;"]
        RagNode["rag node"]
        WritingNode["writing node"]
        RejectNode["reject node\n(LLM 호출 없이 고정 메시지)"]
    end

    subgraph RagAgent["app/agents/rag_agent.py — LangGraph"]
        RagPlanner["rag_planner"]
        Decompose["decompose\n(sub-query 2~3개 생성)"]
        Retrieve["retrieve"]
        Grade["grade_documents\n(관련성 yes/no)"]
        Rewrite["rewrite_query\n(최대 2회 재시도)"]
        Generate["generate"]
    end

    subgraph Retriever["app/retriever.py"]
        BM25["BM25Retriever\n(한국어 토크나이저)"]
        Dense["Qdrant Dense\n(MMR, text-embedding-3-large)"]
        Ensemble["EnsembleRetriever\n(BM25 0.5 + Dense 0.5, RRF c=60)"]
    end

    subgraph WritingAgent["app/agents/writing_agent.py"]
        Router["_router_llm\n(scenario/recipient 식별)"]
        DocMap[("data/required_documents.json\n결정론적 서류 매핑")]
        EmailGen["이메일 본문 생성\n(LLM, 서류목록은 JSON 주입)"]
    end

    subgraph Store["벡터 스토어"]
        Qdrant[("Qdrant\nreal_estatee_manual")]
    end

    FE -->|"질문 제출"| CreateJob
    FE -->|"폴링(2s)"| ReadJob
    FE -.->|"미사용 경로"| Ask

    CreateJob --> Jobs
    Jobs -->|"캐시 히트?"| Cache
    Cache -->|"hit → 즉시 done 기록"| DDB
    Jobs -->|"cache miss"| DDB
    Jobs -->|"job_id 발행"| SQS

    SQS -->|"수신"| Worker
    Worker -->|"user_input 조회"| DDB
    Worker --> RunAssistant["run_assistant()"]
    Worker -->|"결과 write + 캐시 저장"| DDB
    Worker -->|"결과 write"| Cache
    Worker -->|"메시지 삭제"| SQS

    Ask --> RunAssistant
    RunAssistant --> Planner
    Planner -->|"next=rag"| RagNode
    Planner -->|"next=writing"| WritingNode
    Planner -->|"next=reject"| RejectNode

    RagNode --> RagPlanner
    RagPlanner -->|"use_decomp=true"| Decompose
    RagPlanner -->|"use_decomp=false"| Retrieve
    Decompose --> Retrieve
    Retrieve -->|"hybrid_search"| Ensemble
    Ensemble --> BM25
    Ensemble --> Dense
    Dense --> Qdrant
    Retrieve --> Grade
    Grade -->|"no & retry<2"| Rewrite
    Rewrite --> Retrieve
    Grade -->|"yes 또는 retry 소진"| Generate

    WritingNode --> Router
    Router --> DocMap
    DocMap --> EmailGen

    ReadJob --> DDB

    style RejectNode fill:#3a2020
    style Cache fill:#1a3a2a
    style DocMap fill:#1a3a2a
```

```mermaid
flowchart LR
    User["사용자 질문"] --> API["FastAPI\n(Job Queue)"]
    API --> Supervisor["Supervisor\n(라우팅)"]

    Supervisor -->|"절차/서류 질문"| RAG["RAG Agent\n(하이브리드 검색 → 답변)"]
    Supervisor -->|"이메일 작성"| Writing["Writing Agent\n(서류 매핑 → 이메일)"]
    Supervisor -->|"무관한 질문"| Reject["거절 메시지"]

    RAG --> Answer["답변"]
    Writing --> Answer
    Reject --> Answer
    Answer --> User
```

# 캐시 FQA
- cache.get_cached(user_input)으로 DynamoDB의 realestate-cache 테이블을 조회 (질문을 정규화 후 SHA256 해시로 키 삼음
- supervisor는 캐시 존재 모름

```mermaid
flowchart LR
    Q["사용자 질문"] --> J["jobs.submit_job()"]
    J --> C{"캐시 히트?\n(DynamoDB\nrealestate-cache)"}
    C -->|"hit"| D["즉시 done 기록\n(Supervisor 미실행)"]
    C -->|"miss"| S["SQS 큐 발행"]
    S --> W["워커가 폴링"]
    W --> R["run_assistant()\n= Supervisor 실행"]
    R --> Store["결과 저장 +\n캐시에도 저장"]
```